In [0]:
df_bronze = spark.table("irctc_catalog.bronze.train_raw")

In [0]:
display(df_bronze)

In [0]:
df_bronze.printSchema()

In [0]:
print(df_bronze.count())

In [0]:
from pyspark.sql.functions import *

df_bronze.select(
    [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_bronze.columns
    ]).display()

In [0]:
total_count = df_bronze.count()

distinct_count = df_bronze.dropDuplicates().count()

print(f"Total Records: {total_count}")
print(f"Distinct Records: {distinct_count}")
print(f"Duplicate Records: {total_count - distinct_count}")

In [0]:
from pyspark.sql.functions import col

df_silver = df_bronze.select(
    col("route"),
    col("trainName"),
    col("trainNumber"),
    col("runningDays.MON").alias("MON"),
    col("runningDays.TUE").alias("TUE"),
    col("runningDays.WED").alias("WED"),
    col("runningDays.THU").alias("THU"),
    col("runningDays.FRI").alias("FRI"),
    col("runningDays.SAT").alias("SAT"),
    col("runningDays.SUN").alias("SUN"),
    col("trainRoute")
)

In [0]:
display(df_silver)
df_silver.printSchema()

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("irctc_catalog.silver.train_silver")

In [0]:
spark.table("irctc_catalog.silver.train_silver").count()

In [0]:
from pyspark.sql.functions import *

df_route = df_silver.withColumn(
    "route_station",
    explode(col("trainRoute"))
)
display(df_route)

In [0]:
df_route_flat = df_route.select(
    col("trainNumber"),
    col("trainName"),
    col("route"),
    col("route_station.sno").alias("station_seq"),
    col("route_station.stationName").alias("station_name"),
    col("route_station.arrives").alias("arrival_time"),
    col("route_station.departs").alias("departure_time"),
    col("route_station.day").alias("journey_day"),
    col("route_station.distance").alias("distance")
)

In [0]:
display(df_route_flat)

df_route_flat.printSchema()

In [0]:
df_route_flat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("irctc_catalog.silver.train_route_silver")

In [0]:
spark.table(
    "irctc_catalog.silver.train_route_silver"
).count()

In [0]:
display(df_silver)

In [0]:
spark.table(
    "irctc_catalog.silver.train_route_silver"
).printSchema()

#### Silver table 

In [0]:
display(
    spark.table("irctc_catalog.silver.train_route_silver")
)